# Week 102 - Snowpark & Python
https://www.frostyfri.day/en/challenges/blog/2024/07/19/week-102-snowpark-python

## お題
Snowpark Pandas を用いて小売トランザクションデータを分析します。
お題は次の4つ：

1. 1日のうち、売上の大半が発生しているのは何時台か？
2. 最も多く売り上げた店員は誰か？
3. 最初の5商品について、税として20%を差し引いた場合、合計金額はどうなるか？
4. レジ番号4とレジ番号5を統合した場合、最大のレジ番号はいくつになるか？


## よもやま
Snowpark Pandas API は、2024 Snowflake Data Cloud Summit でパブリックプレビューが公開されました。

https://docs.snowflake.com/ja/release-notes/2024/june-summit

この問題はSummit直後の 2024/07/19 に公開された問題です。
「 _ついに、分散型Snowflakeシステム上で馴染みのある構文を使用することで、Pandasに対する愛憎入り混じった関係が完全に愛に満ちたものへと変わります。_ （機械翻訳）」とありますが、このリリースまで Snowpark の構文は PySpark ライクな構文のみでした。

Pandas の構文と Snowpark（PySpark） の構文にはなかなかの違いがあります。
どのくらい違うかというと、、、（後述）

さらに、Pandas と Snowpark（PySpark）には、単一インスタンスで処理するか複数に分散可能か、という違いもあります。  
Pandas でデータ処理を書くと、クライアント側マシンのメモリにデータが展開されてしまう、、、（メモリに乗らないのが目に見えている）  
せっかくそこに Snowflake があるのに、分散しないなんて、もったいない！ [^1]
ということで、ユーザーが Snowflake のコンピューティングリソースをちゃんと使える形で Python でのデータ処理を書きたい！と思うと、Snowpark の構文に向き合わざるを得なかったんですね。なので Pandas が使えることは、冒頭の文章が出てきてしまうくらい、嬉しい話なわけです。  

[^1]: Snowflake のウェアハウスやコンピュートプールであれば、まだ高メモリタイプがあるんですけどね

## セットアップ手順

以下の SQL を実行して、CSV データを参照するための外部 stage を作成します。

```sql
create or replace stage frosty_stage url = 's3://frostyfridaychallenges/challenge_102/';
```

## ノートブック初期化

ノートブックの先頭で、以下を実行して Snowpark Pandas のセッションと DataFrame を準備します。

```python
import modin.pandas as pd
import snowflake.snowpark.modin.plugin
from snowflake.snowpark.context import get_active_session
session = get_active_session()
clothes_shop_df = pd.read_csv('@frosty_stage/clothes_shop_purchases.csv')
```

## 回答すべき4つの問い

1. 1日のうち、売上の大半が発生しているのは何時台か？
2. 最も多く売り上げた店員は誰か？
3. 最初の5商品について、税として20%を差し引いた場合、合計金額はどうなるか？
4. レジ番号4とレジ番号5を統合した場合、最大のレジ番号はいくつになるか？

## 期待される出力

上記4つの問いに対応する4つの回答（原文では回答例がスクリーンショットとして提示されています）。


1. の回答  
   → 10
2. の回答  
   → Mary Harris
3. の回答  
   ```  
   # Transaction_ID Total_Price Total_Price_After_Tax
   0 f0de6fab-a5a3-46cd-9e84-2146862d501c 596.84 477.472
   1 7aa5fe1d-b735-4632-9595-26757711b751 1325.44 1060.352
   2 2a550637-526c-471d-811d-35b6d6413d51 347.22 277.776
   3 cd28441e-845a-4eed-b457-152141b54b0a 570.78 456.624
   4 6d375503-5021-481b-8bf8-9227eece5e22 450.94 360.752
   ```  
4. の回答  
   ```
   # Till Number
   2 21247.39
   4 19597.72
   3 19416.23
   1 17722.92
   ```

## 準備

In [ ]:
use role sysadmin;
use database M_KAJIYA_FROSTY_FRIDAY;
create schema if not exists FF102;
use schema FF102;

In [ ]:
%%sql -r dataframe_2
create or replace stage frosty_stage url = 's3://frostyfridaychallenges/challenge_102/';
ls @frosty_stage;

あらあら、S3を参照できませんでした。ではここでオリチャー発動。それっぽく作ったcsvファイルをWorkspacesに配置します

In [ ]:
! ls -l *.csv

In [ ]:
# 初回はインストール
! pip install "snowflake-snowpark-python[modin]"

お題の準備スクリプトを実行。

In [ ]:
import modin.pandas as pd
import snowflake.snowpark.modin.plugin

from snowflake.snowpark.context import get_active_session


session = get_active_session()

# clothes_shop_df = pd.read_csv('@frosty_stage/clothes_shop_purchases.csv') -- なくしちゃったので
clothes_shop_df = pd.read_csv('clothes_shop_purchases.csv')

In [ ]:
clothes_shop_df

## 解答（1） Snowpark Pandas API


### 1. 1日のうち、売上の大半が発生しているのは何時台か？


In [ ]:
# まず時刻を追加する
clothes_shop_df_1 = clothes_shop_df.copy() # = は Shallow Copy だったので Deep Copy する

clothes_shop_df_1['Hour'] = pd.to_datetime(
        clothes_shop_df_1['Timestamp']
    ).dt.strftime('%H')
# clothes_shop_df_1[['Timestamp', 'Hour']].head()

# 最大値にあたる行にある列の値をとる
clothes_shop_df_1.groupby('Hour')['Total_Price'].sum().idxmax()

### 2. 最も多く売り上げた店員は誰か？

In [ ]:
#1 とやることは同じ
clothes_shop_df.groupby('Clerk_Name')['Total_Price'].sum().idxmax() # 店員ごとの売上値合計


### 3. 最初の5商品について、税として20%を差し引いた場合、合計金額はどうなるか？

In [ ]:
clothes_shop_df_3 = clothes_shop_df.copy()

clothes_shop_df_3['Total_Price_After_Tax'] = clothes_shop_df_3['Total_Price']*0.8 
clothes_shop_df_3[['Transaction_ID', 'Total_Price', 'Total_Price_After_Tax']].head(5)


### 4. レジ番号4とレジ番号5を統合した場合、最大のレジ番号はいくつになるか？

In [ ]:
clothes_shop_df_4 = clothes_shop_df.copy()

# もとは5レジあります
# clothes_shop_df_4.groupby('Till_Number').sum('Total_Price')

# 5 -> 4 にしちゃう
# clothes_shop_df_4.loc[clothes_shop_df_4['Till_Number'] == 5, 'Till_Number']
clothes_shop_df_4.loc[clothes_shop_df_4['Till_Number'] == 5, 'Till_Number'] = 4
clothes_shop_df_4.groupby('Till_Number')[['Total_Price']].sum().sort_values('Total_Price', ascending=False)

## 解答（2）Pandas

この Snowpark Pandas がどれだけ嬉しいかというと、本当に Pandas と構文が同じなんですね。見てみましょう


In [ ]:
import pandas as pd # Pandas に変更
# import snowflake.snowpark.modin.plugin # プラグインはいらないよ

# 全く同じ構文
clothes_shop_df = pd.read_csv('clothes_shop_purchases.csv')

### 1. 1日のうち、売上の大半が発生しているのは何時台か？


In [ ]:
# まず時刻を追加する
clothes_shop_df_1 = clothes_shop_df.copy()

clothes_shop_df_1['Hour'] = pd.to_datetime(
        clothes_shop_df_1['Timestamp']
    ).dt.strftime('%H')
# clothes_shop_df_1[['Timestamp', 'Hour']].head()

# 最大値にあたる行にある列の値をとる
clothes_shop_df_1.groupby('Hour')['Total_Price'].sum().idxmax()

### 2. 最も多く売り上げた店員は誰か？

In [ ]:
#1 とやることは同じ
clothes_shop_df.groupby('Clerk_Name')['Total_Price'].sum().idxmax() # 店員ごとの売上値合計


### 3. 最初の5商品について、税として20%を差し引いた場合、合計金額はどうなるか？

In [ ]:
clothes_shop_df_3 = clothes_shop_df.copy()

clothes_shop_df_3['Total_Price_After_Tax'] = clothes_shop_df_3['Total_Price']*0.8 
clothes_shop_df_3[['Transaction_ID', 'Total_Price', 'Total_Price_After_Tax']].head(5)


### 4. レジ番号4とレジ番号5を統合した場合、最大のレジ番号はいくつになるか？

In [ ]:
clothes_shop_df_4 = clothes_shop_df.copy()

# 5 -> 4 にしちゃう
# clothes_shop_df_4.loc[clothes_shop_df_4['Till_Number'] == 5, 'Till_Number']
clothes_shop_df_4.loc[clothes_shop_df_4['Till_Number'] == 5, 'Till_Number'] = 4
clothes_shop_df_4.groupby('Till_Number')[['Total_Price']].sum().sort_values('Total_Price', ascending=False)

## 解答（3）Snowpark

「 _ついに、分散型Snowflakeシステム上で馴染みのある構文を使用することで、Pandasに対する愛憎入り混じった関係が完全に愛に満ちたものへと変わります。_ 」 どうしてそこまで言われなきゃあいけないのか、まずは試してみましょうか。

In [ ]:
import snowflake.snowpark as snowpark
from snowflake.snowpark.functions import col
from snowflake.snowpark.types import StructType, StructField, StringType, IntegerType, FloatType

schema = StructType([
    StructField("TRANSACTION_ID", StringType()),
    StructField("TIMESTAMP", StringType()),
    StructField("TILL_NUMBER", IntegerType()),
    StructField("CLERK_NAME", StringType()),
    StructField("PRODUCT_NAME", StringType()),
    StructField("TOTAL_PRICE", FloatType())
])

clothes_shop_df = session.read.options(
    {"field_delimiter": ",", "skip_header": 1}
).schema(schema).csv('@FROSTY_STAGE/clothes_shop_purchases.csv') # Stage からしか読めないよ

In [ ]:
clothes_shop_df.show()

### 1. 1日のうち、売上の大半が発生しているのは何時台か？


In [ ]:
from snowflake.snowpark.functions import col, sum as sum_, hour, to_timestamp

clothes_shop_df.with_column(
    "HOUR", hour(to_timestamp(col("TIMESTAMP")))
).group_by("HOUR").agg(
    sum_(col("TOTAL_PRICE")).alias("TOTAL_SALES_BY_HOUR")
).sort(
    col("TOTAL_SALES_BY_HOUR").desc()
).collect()[0][0]

### 2. 最も多く売り上げた店員は誰か？

In [ ]:
clothes_shop_df.group_by("CLERK_NAME").agg(
    sum_(col("TOTAL_PRICE")).alias("TOTAL_SALES_BY_CLERK")
).sort(
    col("TOTAL_SALES_BY_CLERK").desc()
).collect()[0][0]


### 3. 最初の5商品について、税として20%を差し引いた場合、合計金額はどうなるか？

In [ ]:
clothes_shop_df.withColumn(
    "TOTAL_PRICE_AFTER_TAX",
    col("TOTAL_PRICE")*0.8
).select(
    "TRANSACTION_ID",
    "TOTAL_PRICE",
    "TOTAL_PRICE_AFTER_TAX"
).show(5)


### 4. レジ番号4とレジ番号5を統合した場合、最大のレジ番号はいくつになるか？

In [ ]:
from snowflake.snowpark.functions import when, col, lit

clothes_shop_df.withColumn(
    "TILL_NUMBER",
    when(
        col("TILL_NUMBER") == 5, lit(4)
    ).otherwise(
        col("TILL_NUMBER")
    )
).groupBy(
    "TILL_NUMBER"
).agg(
    sum_(col("TOTAL_PRICE")).alias("TOTAL_SALES_BY_TILL_NUMBER")
).select(
    "TILL_NUMBER",
    "TOTAL_SALES_BY_TILL_NUMBER"
).sort(
    col("TOTAL_SALES_BY_TILL_NUMBER").desc()
)

## おまけ

「Pandas ではクライアント側のメモリにデータが展開されてしまう...」といった話をしましたね。
どういうことか、ちょっとdataframeを使って確認してみましょう。

In [ ]:
import sys
import pandas as pd

df = pd.read_csv('clothes_shop_purchases.csv')
print(f"Pandas Dataframe: {df.memory_usage(deep=True).sum() / 1024} KB")
# print(f"(getsizeof: {sys.getsizeof(df) / 1024} KB)") # 同じ結果だよ

In [ ]:
import modin.pandas as pd

df = pd.read_csv('clothes_shop_purchases.csv')
print(f"Snowpark Pandas Dataframe: {df.memory_usage(deep=True).sum() / 1024} KB")

```
UserWarning: Snowpark pandas now runs with hybrid execution enabled by default, and will perform certain operations on smaller data using local, in-memory pandas. To disable this behavior and force all computations to occur in Snowflake, run this line:
from modin.config import AutoSwitchBackend; AutoSwitchBackend.disable()
```

そういえば、ハイブリッド実行モードが出てましたね。

https://docs.snowflake.com/ja/release-notes/clients-drivers/snowpark-python-2025#id17

In [ ]:
# from modin.config import AutoSwitchBackend
# AutoSwitchBackend.disable()

df = session.table("SNOWFLAKE_SAMPLE_DATA.TPCDS_SF100TCL.ITEM").to_snowpark_pandas()

print(f"Snowpark Pandas Dataframe: {df.memory_usage(deep=True).sum() / 1024} KB")

In [ ]:
import snowflake.snowpark as snowpark

df = session.table("SNOWFLAKE_SAMPLE_DATA.TPCDS_SF100TCL.ITEM")
print(f"Snowpark Dataframe: {sys.getsizeof(df) / 1024} KB")